# Baselines Functions (Traditional SHAP + IF)

> 本笔记本只包含 baseline 方法函数定义，不执行训练或评估流程。

In [1]:
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

### _unpack_batch

> 作用：兼容字典批次与元组批次，统一输出 dense/sparse/y/sample_id。

In [2]:
def _unpack_batch(batch):
    if isinstance(batch, dict):
        dense = batch["dense"]
        sparse = batch["sparse"]
        y = batch["labels"]
        sid = batch["sample_id"]
        return dense, sparse, y, sid
    dense, sparse, y, sid = batch
    return dense, sparse, y, sid

### _forward_logits

> 作用：兼容模型输出为单 logit 或 (logit, ... ) 的情况。

In [3]:
def _forward_logits(model, dense: torch.Tensor, sparse: Dict[str, torch.Tensor]) -> torch.Tensor:
    out = model(dense, sparse)
    if isinstance(out, tuple):
        out = out[0]
    return out.view(-1)

### _bce_loss

> 作用：统一使用 BCE with logits 作为 IF 梯度目标。

In [4]:
def _bce_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return F.binary_cross_entropy_with_logits(logits.view(-1), y.float().view(-1))

### _flat_grads

> 作用：将参数梯度展平成单个向量，便于做 IF 内积。

In [5]:
def _flat_grads(grads: List[Optional[torch.Tensor]]) -> torch.Tensor:
    parts = [g.reshape(-1) for g in grads if g is not None]
    if not parts:
        return torch.zeros(1)
    return torch.cat(parts)

### _params_requiring_grad

> 作用：提取需要梯度的模型参数。

In [6]:
def _params_requiring_grad(model) -> List[torch.nn.Parameter]:
    return [p for p in model.parameters() if p.requires_grad]

### _batch_grad_vector

> 作用：计算单个批次损失对参数的梯度向量。

In [7]:
def _batch_grad_vector(model, batch, device: str = DEVICE) -> torch.Tensor:
    dense, sparse, y, _sid = _unpack_batch(batch)
    dense = dense.to(device)
    sparse = {k: v.to(device) for k, v in sparse.items()}
    y = y.to(device)
    logits = _forward_logits(model, dense, sparse)
    loss = _bce_loss(logits, y)
    params = _params_requiring_grad(model)
    grads = torch.autograd.grad(loss, params, retain_graph=False, create_graph=False, allow_unused=True)
    return _flat_grads([g.detach().cpu() if g is not None else None for g in grads])

### _hvp

> 作用：计算 Hessian-vector product，供 LiSSA 近似 $H^{-1}v$。

In [8]:
def _hvp(loss: torch.Tensor, params: List[torch.nn.Parameter], vec: List[torch.Tensor]) -> List[torch.Tensor]:
    g1 = torch.autograd.grad(loss, params, create_graph=True, retain_graph=True, allow_unused=True)
    dot = torch.tensor(0.0, device=loss.device)
    for gi, vi in zip(g1, vec):
        if gi is None:
            continue
        dot = dot + (gi * vi).sum()
    g2 = torch.autograd.grad(dot, params, retain_graph=False, allow_unused=True)
    out = []
    for hi, p in zip(g2, params):
        if hi is None:
            out.append(torch.zeros_like(p))
        else:
            out.append(hi)
    return out

### _inverse_hvp_lissa

> 作用：用 LiSSA 递推近似 $H^{-1}v$。

In [9]:
def _inverse_hvp_lissa(
    model,
    train_loader_eval: DataLoader,
    v: List[torch.Tensor],
    recursion_depth: int = 200,
    damping: float = 0.01,
    scale: float = 10.0,
    device: str = DEVICE,
) -> List[torch.Tensor]:
    params = _params_requiring_grad(model)
    cur_estimate = [vi.detach().clone() for vi in v]
    model.train()

    it = 0
    for batch in train_loader_eval:
        dense, sparse, y, _sid = _unpack_batch(batch)
        dense = dense.to(device)
        sparse = {k: t.to(device) for k, t in sparse.items()}
        y = y.to(device)
        logits = _forward_logits(model, dense, sparse)
        loss = _bce_loss(logits, y)

        hv = _hvp(loss, params, cur_estimate)
        new_estimate = []
        for vi, ci, hi in zip(v, cur_estimate, hv):
            new_ci = vi + (1.0 - damping) * ci - hi / scale
            new_estimate.append(new_ci)
        cur_estimate = new_estimate

        it += 1
        if it >= recursion_depth:
            break

    return [c / scale for c in cur_estimate]

### compute_if_baseline

> 作用：计算传统 Influence Function baseline，输出 `sample_id -> influence_score`。

In [10]:
def compute_if_baseline(
    model,
    train_loader_eval: DataLoader,
    val_loader: DataLoader,
    max_samples: int = 256,
    recursion_depth: int = 200,
    damping: float = 0.01,
    scale: float = 10.0,
    device: str = DEVICE,
) -> Dict[int, float]:
    model = model.to(device)
    params = _params_requiring_grad(model)

    # 1) 验证集平均梯度 g_val
    val_grads = [torch.zeros_like(p, device=device) for p in params]
    n_val = 0
    for batch in val_loader:
        dense, sparse, y, _sid = _unpack_batch(batch)
        dense = dense.to(device)
        sparse = {k: t.to(device) for k, t in sparse.items()}
        y = y.to(device)
        logits = _forward_logits(model, dense, sparse)
        loss = _bce_loss(logits, y)
        grads = torch.autograd.grad(loss, params, retain_graph=False, create_graph=False, allow_unused=True)
        for i, g in enumerate(grads):
            if g is not None:
                val_grads[i] = val_grads[i] + g.detach()
        n_val += 1
    if n_val == 0:
        return {}
    val_grads = [g / float(n_val) for g in val_grads]

    # 2) s_test = H^{-1} g_val
    s_test = _inverse_hvp_lissa(
        model=model,
        train_loader_eval=train_loader_eval,
        v=val_grads,
        recursion_depth=recursion_depth,
        damping=damping,
        scale=scale,
        device=device,
    )
    s_flat = _flat_grads([t.detach().cpu() for t in s_test])

    # 3) 每个训练样本 influence = - grad(z)^T s_test
    scores = {}
    c = 0
    for batch in train_loader_eval:
        dense, sparse, y, sid = _unpack_batch(batch)
        bs = dense.shape[0]
        for i in range(bs):
            one_batch = (
                dense[i : i + 1],
                {k: v[i : i + 1] for k, v in sparse.items()},
                y[i : i + 1],
                sid[i : i + 1],
            )
            g_flat = _batch_grad_vector(model, one_batch, device=device)
            if g_flat.numel() == s_flat.numel():
                influence = -float(torch.dot(g_flat, s_flat).item())
            else:
                influence = 0.0
            scores[int(sid[i].item())] = influence
            c += 1
            if c >= max_samples:
                return scores
    return scores

### prepared_to_tabular

> 作用：将 PreparedData 转换为表格特征矩阵，供 SHAP baseline 使用。

In [11]:
def prepared_to_tabular(prepared, max_samples: Optional[int] = None):
    n = prepared.labels.shape[0]
    idx = np.arange(n)
    if max_samples is not None and max_samples > 0 and max_samples < n:
        idx = np.random.choice(idx, size=max_samples, replace=False)

    dense = prepared.dense[idx].detach().cpu().numpy().astype(np.float32)
    sparse_fields = list(prepared.sparse_fields)
    sparse_cols = []
    for f in sparse_fields:
        sparse_cols.append(prepared.sparse[f][idx].detach().cpu().numpy().astype(np.float32).reshape(-1, 1))
    sparse_arr = np.concatenate(sparse_cols, axis=1) if sparse_cols else np.zeros((len(idx), 0), dtype=np.float32)

    x = np.concatenate([dense, sparse_arr], axis=1)
    feature_names = list(prepared.dense_fields) + [f"sparse::{f}" for f in sparse_fields]
    sample_ids = prepared.sample_ids[idx].detach().cpu().numpy().astype(np.int64)
    dense_dim = dense.shape[1]
    return x, sample_ids, feature_names, dense_dim, sparse_fields

### _predict_from_tabular

> 作用：把表格特征还原为 dense/sparse 并调用模型，返回概率。

In [12]:
def _predict_from_tabular(
    model,
    x: np.ndarray,
    dense_dim: int,
    sparse_fields: List[str],
    device: str = DEVICE,
    batch_size: int = 1024,
) -> np.ndarray:
    model = model.to(device)
    model.eval()
    outs = []
    with torch.no_grad():
        for st in range(0, x.shape[0], batch_size):
            ed = min(st + batch_size, x.shape[0])
            xb = x[st:ed]
            dense = torch.tensor(xb[:, :dense_dim], dtype=torch.float32, device=device)
            sparse = {}
            for i, f in enumerate(sparse_fields):
                sparse[f] = torch.tensor(np.clip(xb[:, dense_dim + i], 0, None).astype(np.int64), dtype=torch.long, device=device)
            logits = _forward_logits(model, dense, sparse)
            outs.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(outs, axis=0)

### _permutation_shap_fallback

> 作用：当 `shap` 包不可用时，用置换近似得到传统 SHAP 风格分数。

In [13]:
def _permutation_shap_fallback(
    model,
    x_eval: np.ndarray,
    x_bg: np.ndarray,
    dense_dim: int,
    sparse_fields: List[str],
    num_permutations: int = 20,
    device: str = DEVICE,
) -> np.ndarray:
    rng = np.random.RandomState(42)
    n, d = x_eval.shape
    phi = np.zeros((n, d), dtype=np.float64)
    bg_mean = x_bg.mean(axis=0, keepdims=True)

    for _ in range(num_permutations):
        order = rng.permutation(d)
        x_cur = np.repeat(bg_mean, repeats=n, axis=0)
        p_prev = _predict_from_tabular(model, x_cur, dense_dim, sparse_fields, device=device)
        for j in order:
            x_next = x_cur.copy()
            x_next[:, j] = x_eval[:, j]
            p_next = _predict_from_tabular(model, x_next, dense_dim, sparse_fields, device=device)
            phi[:, j] += (p_next - p_prev)
            x_cur = x_next
            p_prev = p_next

    phi = phi / float(max(num_permutations, 1))
    return phi

### compute_shap_baseline

> 作用：计算传统 SHAP baseline（优先 Kernel SHAP，失败时回退置换近似）。

In [14]:
def compute_shap_baseline(
    model,
    prepared,
    background_size: int = 128,
    eval_size: int = 256,
    nsamples: int = 100,
    device: str = DEVICE,
) -> Dict[int, float]:
    x, sample_ids, feature_names, dense_dim, sparse_fields = prepared_to_tabular(prepared, max_samples=None)
    if x.shape[0] == 0:
        return {}

    rng = np.random.RandomState(42)
    bg_idx = rng.choice(np.arange(x.shape[0]), size=min(background_size, x.shape[0]), replace=False)
    ev_idx = rng.choice(np.arange(x.shape[0]), size=min(eval_size, x.shape[0]), replace=False)
    x_bg = x[bg_idx]
    x_ev = x[ev_idx]
    sid_ev = sample_ids[ev_idx]

    phi = None
    try:
        import shap

        def _pred_fn(arr):
            arr = np.asarray(arr, dtype=np.float32)
            return _predict_from_tabular(model, arr, dense_dim, sparse_fields, device=device)

        explainer = shap.KernelExplainer(_pred_fn, x_bg)
        shap_vals = explainer.shap_values(x_ev, nsamples=nsamples)
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[0]
        phi = np.asarray(shap_vals, dtype=np.float64)
    except Exception:
        phi = _permutation_shap_fallback(
            model=model,
            x_eval=x_ev,
            x_bg=x_bg,
            dense_dim=dense_dim,
            sparse_fields=sparse_fields,
            num_permutations=20,
            device=device,
        )

    score_arr = np.abs(phi).sum(axis=1)
    scores = {int(sid): float(sc) for sid, sc in zip(sid_ev, score_arr)}
    return scores

### scores_to_frame

> 作用：将 baseline 分数字典转为 DataFrame，便于导出与对比。

In [15]:
def scores_to_frame(scores: Dict[int, float], score_name: str) -> pd.DataFrame:
    rows = [{"sample_id": int(k), score_name: float(v)} for k, v in scores.items()]
    if not rows:
        return pd.DataFrame(columns=["sample_id", score_name])
    return pd.DataFrame(rows).sort_values(score_name, ascending=False).reset_index(drop=True)